# Missing Data — NaN / NA

## Definition
Real-world datasets almost always have **missing values**. Pandas uses `NaN` (Not a Number) for numeric columns
and `pd.NA` / `None` for other types to represent missing data.

### Why Missing Data Matters in ML
- Most ML algorithms **cannot handle NaN** — sklearn will raise an error
- Wrong imputation strategy can **introduce data leakage**
- Understanding WHY data is missing matters:
  - **MCAR** — Missing Completely At Random (safe to drop)
  - **MAR** — Missing At Random (impute based on other features)
  - **MNAR** — Missing Not At Random (need domain knowledge)

### Key Functions
| Function | Description |
|----------|-------------|
| `df.isna()` / `df.isnull()` | Boolean mask of NaN locations |
| `df.notna()` | Boolean mask of non-NaN |
| `df.isna().sum()` | Count of NaN per column |
| `df.isna().mean()` | Fraction of NaN per column |
| `df.dropna()` | Remove rows/cols with NaN |
| `df.fillna()` | Fill NaN with values/strategy |
| `df.interpolate()` | Interpolate NaN values |

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
df = pd.DataFrame({
    'age':      [25, np.nan, 30, 22, np.nan, 35, 28],
    'income':   [50000, 65000, np.nan, 48000, 72000, np.nan, 55000],
    'score':    [85.0, 78.0, 92.0, np.nan, 88.0, 76.0, np.nan],
    'category': ['A', 'B', None, 'A', 'C', 'B', None],
    'label':    [1, 0, 1, 1, 0, 1, 0]
})
print(df)

In [ ]:
# Detecting missing values
print('isna() mask:')
print(df.isna())
print()
print('Missing count per column:')
print(df.isna().sum())
print()
print('Missing percentage per column:')
print((df.isna().mean() * 100).round(1))

In [ ]:
# Total missing values
total_missing = df.isna().sum().sum()
total_cells = df.size
print(f'Total missing: {total_missing} / {total_cells} ({100*total_missing/total_cells:.1f}%)')

# Rows with ANY missing value
print('\nRows with any NaN:')
print(df[df.isna().any(axis=1)])

# Rows with ALL values missing
print('\nRows with all NaN:')
print(df[df.isna().all(axis=1)])

In [ ]:
# dropna() — removing missing data
# Drop rows with ANY NaN
print('dropna() — any NaN:', df.dropna().shape)

# Drop rows with ALL NaN
print('dropna(how=all):', df.dropna(how='all').shape)

# Drop rows with NaN in specific columns
print('dropna(subset=[age]):', df.dropna(subset=['age']).shape)

# Drop columns with too many NaN (>30% missing)
threshold = int(len(df) * 0.7)  # keep cols with at least 70% non-null
print('dropna thresh (cols):', df.dropna(axis=1, thresh=threshold).shape)

In [ ]:
# fillna() strategies
df_clean = df.copy()

# Fill with constant
df_clean['category'] = df_clean['category'].fillna('Unknown')

# Fill numeric with mean
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].mean())

# Fill with median (more robust to outliers)
df_clean['income'] = df_clean['income'].fillna(df_clean['income'].median())

# Fill with mode (for categorical)
df_clean['score'] = df_clean['score'].fillna(df_clean['score'].mode()[0])

print(df_clean)

In [ ]:
# Forward fill and backward fill (for time-series)
ts = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan, 6.0])
print('Original:', ts.values)
print('ffill:   ', ts.ffill().values)   # propagate last valid forward
print('bfill:   ', ts.bfill().values)   # propagate next valid backward

In [ ]:
# interpolate() — for ordered/time-series data
ts = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan, 6.0])
print('linear interpolate:', ts.interpolate(method='linear').values)
print('polynomial (deg 2):', ts.interpolate(method='polynomial', order=2).round(2).values)

In [ ]:
# EXCEPTION: Operations with NaN propagate
s = pd.Series([1, 2, np.nan, 4])
print('sum (skips NaN by default):', s.sum())
print('mean (skips NaN):', s.mean())
print('min:', s.min())  # pandas ignores NaN in aggregations

# But comparison with NaN is always False (like NumPy)
print('nan == nan:', np.nan == np.nan)  # False
# Correct way to check NaN
print('Use isna():', pd.isna(np.nan))